In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Data import and analysis

In [3]:
df = pd.read_csv('../dataset/dirty_cafe_sales.csv')

In [45]:
df


,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11
...,...,...,...,...,...,...,...,...
9995,TXN_7672686,Coffee,2,2.0,4.0,NaN,UNKNOWN,2023-08-30
9996,TXN_9659401,NaN,3,NaN,3.0,Digital Wallet,NaN,2023-06-02
9997,TXN_5255387,Coffee,4,2.0,8.0,Digital Wallet,NaN,2023-03-02
9998,TXN_7695629,Cookie,3,NaN,3.0,Digital Wallet,NaN,2023-12-02


In [4]:
# Info
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   Transaction ID    10000 non-null  str  
 1   Item              9667 non-null   str  
 2   Quantity          9862 non-null   str  
 3   Price Per Unit    9821 non-null   str  
 4   Total Spent       9827 non-null   str  
 5   Payment Method    7421 non-null   str  
 6   Location          6735 non-null   str  
 7   Transaction Date  9841 non-null   str  
dtypes: str(8)
memory usage: 625.1 KB


In [5]:
# Describe
df.describe()

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
count,10000,9667,9862,9821,9827,7421,6735,9841
unique,10000,10,7,8,19,5,4,367
top,TXN_1961373,Juice,5,3.0,6.0,Digital Wallet,Takeaway,UNKNOWN
freq,1,1171,2013,2429,979,2291,3022,159


In [7]:
# Missing values
df.isna().sum()

Transaction ID         0
Item                 333
Quantity             138
Price Per Unit       179
Total Spent          173
Payment Method      2579
Location            3265
Transaction Date     159
dtype: int64

In [73]:
# Item column clenup
cleanup_df = df.copy()

invalid_items = ['UNKNOWN', 'ERROR']

mask = (
        cleanup_df['Item'].isin(invalid_items)
        | cleanup_df['Item'].isna()
)

price_column = pd.to_numeric(cleanup_df['Price Per Unit'], errors='coerce')

rows_to_drop = []

for i in cleanup_df.loc[mask].index:
    row = cleanup_df.loc[i]

    price = pd.to_numeric(row['Price Per Unit'], errors='coerce')

    if pd.isna(price):
        total = pd.to_numeric(row['Total Spent'], errors='coerce')
        quantity = pd.to_numeric(row['Quantity'], errors='coerce')

        if pd.notna(total) and pd.notna(quantity) and quantity != 0:
            price = total / quantity
        else:
            rows_to_drop.append(i)
            continue

    match = cleanup_df[
        (abs(price_column - price) < 0.01)
        & cleanup_df['Item'].notna()
        & (~cleanup_df['Item'].isin(invalid_items))
        & (cleanup_df.index != i)
        ]

    if not match.empty:
        cleanup_df.loc[i, 'Item'] = match.iloc[0]['Item']

cleanup_df = cleanup_df.drop(index=rows_to_drop).reset_index(drop=True)

df = cleanup_df.copy()

In [77]:
# After Item column cleanup check. 2 rows were dropped as there was no way to determine the value that was supposed to be there
df.isna().sum()

Transaction ID         0
Item                   0
Quantity             138
Price Per Unit       177
Total Spent          172
Payment Method      2577
Location            3262
Transaction Date     159
dtype: int64

In [15]:
df.loc[pd.to_numeric(df['Total Spent'], errors='coerce').isna(), 'Total Spent'].unique()

<StringArray>
['ERROR', nan, 'UNKNOWN']
Length: 3, dtype: str

In [19]:
df['Item'].unique() #error values: 'UNKNOWN' 'ERROR', nan

<StringArray>
[  'Coffee',     'Cake',   'Cookie',    'Salad', 'Smoothie',  'UNKNOWN',
 'Sandwich',        nan,    'ERROR',    'Juice',      'Tea']
Length: 11, dtype: str

...